# Advanced Artificial Intelligence Task 2
### Produce classification via pre-trained models across different architectures

- **CNN**:          EfficientNet_V2_S_Weights.IMAGENET1K_V1
- **TRANSFORMER**:   Swin_S_Weights.IMAGENET1K_V1 
- **HYBRID**:       MaxVit_T
Details for the pre-trained weights can be found [here](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.efficientnet_v2_s.html#torchvision.models.efficientnet_v2_s).

View logs with ```tensorboard --logdir task_2/runs```.

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim
import torchmetrics
import os
from torchvision.models import get_model, get_weight
from torch.nn import CrossEntropyLoss
from torch.optim import SGD, Adam
from torchmetrics import Metric
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from pathlib import Path
import datetime
import sys
from sklearn.model_selection import train_test_split
from safetensors.torch import save_file
sys.path.append("..")
sys.path.append(".")
from experiment_configs import task_2_config as experiments
from utils.dataset import ProduceDataset
from utils.mtl_model import MultiTaskClassifier
from utils.utils import seed_everything


In [ ]:
# Load experiment configuration (as defined in experiment_configs/task_2_config.py)
valid_experiments = [
    name for name in dir(experiments) 
    if isinstance(getattr(experiments, name), experiments.Experiment)
]
print(f"Selectable experiments:\n {valid_experiments}")

# Define experiment
EXP = experiments.EX3_EFFICIENTNET_FINETUNE_MTL
print(f"\nSelected: {EXP.display_name}")

Selectable experiments:
 ['EX1_EFFICIENTNET_FREEZE', 'EX2_EFFICIENTNET_FINETUNE', 'EX3_EFFICIENTNET_FINETUNE_MTL']
Selected: EX3_EFFICIENTNET_FINETUNE_MTL


In [ ]:
# TRAINING PARAMETERS
# Define primary dataset path
PRODUCE_DATESET_PATH = Path(".") / "data" / "Fruit_And_Vegetable_Diseases_Dataset"
WORKERS = int(os.cpu_count() * 0.75) 

# Seed for reproducability
# https://gist.github.com/ihoromi4/b681a9088f348942b01711f251e5f964
seed_everything(42)
gen = torch.Generator()
gen.manual_seed(42)

# MTL WEIGHTS
# Task weighting follows a zero-sum logic to maintain loss scale consistency.
# Primary: Binary Health (Healthy/Rotten)
# Auxiliary: Multiclass Produce Type
EXP.mtl_primary_weight = EXP.mtl_primary_weight
TYPE_LOSS_WEIGHT = 1 - EXP.mtl_primary_weight
assert EXP.mtl_primary_weight + TYPE_LOSS_WEIGHT == 1

# Batch size of 32 for all experiemnts
# Change batch size and acc proportion as required for performance
BATCH_SIZE = 16 
ACCUMULATION_STEPS = 2
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * ACCUMULATION_STEPS
assert EFFECTIVE_BATCH_SIZE == 32, ("Effective batch size must be 32 for training stability. "
                                    "Adjust BATCH_SIZE or GRADIENT_ACCUMULATION_STEPS accordingly.")
TEST_SPLIT = 0.8 
MAXIMUM_EPOCHS = 20 # Define maximum for early-stopping.
NUM_CLASSES = 2 # Healthy & Rotten
EARLY_STOPPING_PATIENCE = 5 # Defines how many non-improvement epochs will terminate run

# Load the pre-trained weights 
pretrained_weights = get_weight(EXP.weight_string)
PRETRAINED_MODEL = get_model(EXP.architecture, weights=pretrained_weights)
# Extract required transformations for model image input
auto_transforms = pretrained_weights.transforms()
# print(f"{EXP.weight_string} transforms:\n\n {pretrained_weights.transforms()}")

In [ ]:
# Initialize the dataset and assign labels based on folder structure
produce_dataset = ProduceDataset(dataset_root_dir=PRODUCE_DATESET_PATH,
                                  transform=auto_transforms)
produce_dataset.print_class_balance()
produce_dataset.display_examples(num_samples=5, show_transformed=False)

**Transfer Learning vs. Finetuning**

Both methods implement a classifer head (the final layer) for the new task (Healthy vs. Rotten). They adapt pre-trained model behaviour in distinct ways:
| Method | Description | When to use? |
|---|---|---|
| Transfer learning | Freeze most or all pre-trained weights and train new classifer head. | Small dataset, similar domain and task. |
| Fine-tuning | Update some or all model weights as well as new classifier head.| Large dataset, less similar domain and task |


In [ ]:
# Set transfer learning method
# https://medium.com/@marklpd/transfer-learning-finetuning-for-cnns-in-pytorch-5c4ade873d93
if EXP.training.transfer_type == "FREEZE":
    # Freeze backbone if enabled; gradients are enabled by default for new layers
    for parameters in PRETRAINED_MODEL.parameters():
        # Do not compute backbone gradients (i.e., freeze weights)
        parameters.requires_grad = False 

In [ ]:
# Instantiate MTL model
model = MultiTaskClassifier(PRETRAINED_MODEL, 
                            num_produce_classes=produce_dataset.num_produce_types)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
# Define loss function
health_criterion = nn.CrossEntropyLoss()
type_criterion = nn.CrossEntropyLoss()

In [ ]:
# Define optimiser 
# Stochastic Gradient Descent optimizer
# FREEZE: only train the new MTL heads (backbone is frozen via requires_grad=False)
# FINETUNE: train everything (backbone + heads)
if EXP.training.transfer_type == "FREEZE":
    # Only pass parameters that require gradients (i.e., the two new heads)
    trainable_params = [parameter for parameter in model.parameters() 
                        if parameter.requires_grad]
    optimizer = optim.SGD(trainable_params, lr=EXP.training.learning_rate, momentum=EXP.training.momentum)
else:
    optimizer = optim.SGD(model.parameters(), lr=EXP.training.learning_rate, momentum=EXP.training.momentum)

In [ ]:
# Define learning rate scheduler 
if EXP.scheduler .type == "StepLR":
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=EXP.scheduler .step_size, gamma=EXP.scheduler.gamma)

In [ ]:
# Split dataset
# Stratify to respect the original class distribution between train and eval
train_idx, val_idx = train_test_split(
    range(len(produce_dataset)),
    test_size=1-TEST_SPLIT,
    stratify=produce_dataset.produce_type_lbls,
    random_state=42
)

train_dataset = torch.utils.data.Subset(produce_dataset, train_idx)
val_dataset = torch.utils.data.Subset(produce_dataset, val_idx)

In [ ]:
# Initialise dataloaders - https://www.geeksforgeeks.org/deep-learning/pytorch-dataloader/
# Shuffle training data for better generalization
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                          num_workers=WORKERS, pin_memory=True, generator=gen) 

# No need to shuffle validation data 
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                        num_workers=WORKERS, pin_memory=True, generator=gen) 

print(f"Training size: {len(train_dataset)} images\nValidation size: {len(val_dataset)} images.")

In [ ]:
# Define train/validation functions
# https://medium.com/@ebimsv/mastering-cnns-in-pytorch-week-2-building-and-training-custom-and-pretrained-cnns-for-image-f040572c73c1

class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.sum = 0
        self.count = 0

    def update(self, value, n=1):
        self.sum += value * n
        self.count += n

    @property
    def avg(self):
        return self.sum / self.count if self.count > 0 else 0

def train_one_epoch(model: MultiTaskClassifier, dataloader: DataLoader, 
                    health_criterion: CrossEntropyLoss, type_criterion: CrossEntropyLoss, 
                    optimizer: SGD | Adam, device: torch.device, epoch: int, 
                    health_accuracy: Metric, type_accuracy: Metric) -> tuple:
    model.train()
    loss_meter = AverageMeter()
    health_accuracy.reset()
    type_accuracy.reset()
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1} [Training]", leave=False)

    # zero gradients once at the start of the epoch
    optimizer.zero_grad()

    # Iterate over batches (image, health label, type label)
    for id, (X_batch, y_health, y_type) in enumerate(progress_bar):
        # Move data to device
        X_batch = X_batch.to(device)
        y_health = y_health.to(device)
        y_type = y_type.to(device)

        # Forward pass
        health_output, type_output = model(X_batch)

        # Calculate loss (combined, weighted by defined task weighting)
        loss_health = health_criterion(health_output, y_health)
        loss_type = type_criterion(type_output, y_type)
        loss = (EXP.mtl_primary_weight * loss_health 
            + TYPE_LOSS_WEIGHT * loss_type)

        # Scale loss by accumulation steps so gradients are averaged, not summed
        (loss / ACCUMULATION_STEPS).backward()

        # Only step the optimizer every ACCUMULATION_STEPS batches (or on the last batch)
        if (id + 1) % ACCUMULATION_STEPS == 0 or (id + 1) == len(dataloader):
            optimizer.step()
            optimizer.zero_grad()

        # Update metrics (use unscaled loss for logging)
        loss_meter.update(loss.item(), X_batch.size(0))
        health_preds = health_output.argmax(dim=1)
        type_preds = type_output.argmax(dim=1)
        health_accuracy.update(health_preds, y_health)
        type_accuracy.update(type_preds, y_type)
        if id % 50 == 0:
            progress_bar.set_postfix(loss=loss_meter.avg, health_acc=health_accuracy.compute().item(), type_acc=type_accuracy.compute().item())

    avg_loss = loss_meter.avg
    avg_health_acc = health_accuracy.compute().item()
    avg_type_acc = type_accuracy.compute().item()

    return avg_loss, avg_health_acc, avg_type_acc

def validate(model: MultiTaskClassifier, dataloader: DataLoader, health_criterion: CrossEntropyLoss, 
             type_criterion: CrossEntropyLoss, device: torch.device, epoch: int, 
             health_accuracy: Metric, type_accuracy: Metric) -> tuple:
    model.eval()
    loss_meter = AverageMeter()
    health_accuracy.reset()
    type_accuracy.reset()
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1} [Validation]", leave=False)
    
    # Disable gradient computation for validation
    with torch.no_grad():
        for id, (X_batch, y_health, y_type) in enumerate(progress_bar):
            X_batch = X_batch.to(device)
            y_health = y_health.to(device)
            y_type = y_type.to(device)

            health_output, type_output = model(X_batch)
            loss_health = health_criterion(health_output, y_health)
            loss_type = type_criterion(type_output, y_type)
            loss = (EXP.mtl_primary_weight * loss_health 
                + TYPE_LOSS_WEIGHT * loss_type)

            # Update metrics
            loss_meter.update(loss.item(), X_batch.size(0))
            health_preds = health_output.argmax(dim=1)
            type_preds = type_output.argmax(dim=1)
            health_accuracy.update(health_preds, y_health)
            type_accuracy.update(type_preds, y_type)

            if id % 50 == 0:
                progress_bar.set_postfix(loss=loss_meter.avg, health_acc=health_accuracy.compute().item(), type_acc=type_accuracy.compute().item())
        
    avg_loss = loss_meter.avg
    avg_health_acc = health_accuracy.compute().item()
    avg_type_acc = type_accuracy.compute().item()
    
    return avg_loss, avg_health_acc, avg_type_acc


In [ ]:
# TRAIN/VALIDATION LOOP

writer = SummaryWriter()

# Set accuracy metrics (one per task, per split)
train_health_accuracy = torchmetrics.Accuracy(task="binary", num_classes=NUM_CLASSES).to(device)
train_type_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=produce_dataset.num_produce_types).to(device)
val_health_accuracy = torchmetrics.Accuracy(task="binary", num_classes=NUM_CLASSES).to(device)
val_type_accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=produce_dataset.num_produce_types).to(device)

timestamp = datetime.datetime.now().strftime("%Y%m%d%H%M%S")
model_save_name = f"{EXP.display_name }_{timestamp}"

best_val_loss = float('inf')
patience_counter = 0

for epoch in range(MAXIMUM_EPOCHS):
    train_loss, train_health_acc, train_type_acc = train_one_epoch(model, train_loader, health_criterion, 
                                                                   type_criterion, optimizer, device, epoch, 
                                                                   train_health_accuracy, train_type_accuracy)
    
    val_loss, val_health_acc, val_type_acc = validate(model, val_loader, health_criterion, type_criterion, 
                                                      device, epoch, val_health_accuracy, val_type_accuracy)

    # Step scheduler at each epoch
    lr_scheduler.step()

    # Early stopping on val loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        # Save in safetensor format to safety
        save_file(model.state_dict(), f'models/{model_save_name}.safetensors')
        print(f"Epoch {epoch+1}: New best model saved with Val Loss: {best_val_loss:.4f}")
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

    # Log metrics to TensorBoard  
    writer.add_scalar('Loss/Train', train_loss, epoch)
    writer.add_scalar('Loss/Validation', val_loss, epoch)
    writer.add_scalar('Accuracy/Train_Health', train_health_acc, epoch)
    writer.add_scalar('Accuracy/Train_Type', train_type_acc, epoch)
    writer.add_scalar('Accuracy/Val_Health', val_health_acc, epoch)
    writer.add_scalar('Accuracy/Val_Type', val_type_acc, epoch)

writer.close()
# To view results, run this in terminal: tensorboard --logdir task_2/runs